# Installation Steps
Before running the code, install all required dependencies. These steps ensure Spark, Great Expectations, and related libraries are available in your environment.

In [ ]:
# 1. Install Java (required for Spark)
!apt-get update -y && apt-get install -y openjdk-11-jdk

# 2. Install pyspark
!pip install pyspark

# 3. Install findspark (for Spark integration in notebooks)
!pip install findspark

# 4. Install Great Expectations
!pip install great_expectations

# 5. Install pandas (for batch validation)
!pip install pandas

# 6. Install pyarrow (for Spark to Pandas conversion)
!pip install pyarrow

# 7. Install kafka-python (for Kafka integration/testing)
!pip install kafka-python

# 8. Install requests (for HTTP requests, if needed)
!pip install requests

# 9. Install tqdm (for progress bars, optional)
!pip install tqdm

# 10. Install ipywidgets (for notebook widgets, optional)
!pip install ipywidgets


Get:1 http://deb.debian.org/debian trixie InRelease [140 kB]
Get:2 http://deb.debian.org/debian trixie-updates InRelease [47.3 kB]
Get:3 http://deb.debian.org/debian-security trixie-security InRelease [43.4 kB]
Get:4 http://deb.debian.org/debian trixie/main amd64 Packages [9671 kB]
Get:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages [5412 B] 
Get:6 http://deb.debian.org/debian-security trixie-security/main amd64 Packages [130 kB]
Fetched 10.0 MB in 34s (295 kB/s)                                              
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package openjdk-11-jdk

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 58.5 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 64.1 kB/s eta 0:00:00a 0:00:01
     ━━━━

# PySpark CDC Clickstream Consumer
This notebook demonstrates consuming CDC clickstream data from Kafka using PySpark, with a plan to integrate Great Expectations for data validation.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StructField, StructType, StringType

In [6]:
# Debezium message schema (JSON with schema+payload envelope)
payload_after_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("referrer", StringType(), True),
    StructField("device", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("ip", StringType(), True),
    StructField("product_id", StringType(), True),
])

payload_schema = StructType([
    StructField("before", payload_after_schema, True),
    StructField("after", payload_after_schema, True),
    StructField("op", StringType(), True),
    StructField("ts_ms", StringType(), True),
])

debezium_envelope_schema = StructType([
    StructField("payload", payload_schema, True)
])

In [7]:
def build_spark() -> SparkSession:
    return (
        SparkSession.builder.appName("clickstream-cdc-consumer")
        .master("spark://ingest-spark-master:7077")
        .config(
            "spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1",
        )
        .getOrCreate()
    )

In [8]:
# Build Spark session
spark = build_spark()
spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0e0db055-f68a-44f3-a733-c19eda71332d;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 897ms :: artifacts dl 8ms
	:: modules in use

In [9]:
# Read from Kafka
raw_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka1:9092,kafka2:9092,kafka3:9092")
    .option("subscribe", "srcs_ecommerce.public.clickstream")
    .option("startingOffsets", "latest")
    .load()
)

In [10]:
# Parse Debezium envelope
parsed = raw_stream.select(
    F.from_json(F.col("value").cast("string"), debezium_envelope_schema).alias("msg")
)

In [11]:
# Extract clickstream events
clickstream_events = (
    parsed.select(
        F.col("msg.payload.op").alias("op"),
        F.col("msg.payload.after.id").alias("id"),
        F.col("msg.payload.after.timestamp").alias("event_ts"),
        F.col("msg.payload.after.user_id").alias("user_id"),
        F.col("msg.payload.after.event_type").alias("event_type"),
        F.col("msg.payload.after.page_url").alias("page_url"),
        F.col("msg.payload.after.referrer").alias("referrer"),
        F.col("msg.payload.after.device").alias("device"),
        F.col("msg.payload.after.browser").alias("browser"),
        F.col("msg.payload.after.ip").alias("ip"),
        F.col("msg.payload.after.product_id").alias("product_id"),
    )
    .filter(F.col("id").isNotNull())
    .filter(F.col("op").isin("c", "u", "r"))
)

In [ ]:
# Write stream to console (for debugging/demo)
query = (
    clickstream_events.writeStream.format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option("numRows", "20")
    .start()
)

# query.awaitTermination()

# Great Expectations Integration
This section demonstrates how to validate each micro-batch of clickstream data using Great Expectations.

In [ ]:
import great_expectations as ge
from great_expectations.dataset import PandasDataset
import pandas as pd

def validate_with_ge(batch_df: pd.DataFrame):
    # Convert to Great Expectations dataset
    ge_df = ge.from_pandas(batch_df)
    # Example expectations (customize as needed)
    ge_df.expect_column_values_to_not_be_null("id")
    ge_df.expect_column_values_to_be_in_set("op", ["c", "u", "r"])
    ge_df.expect_column_values_to_not_be_null("event_ts")
    # Validate
    results = ge_df.validate()
    print("Validation results:", results.success)
    if not results.success:
        print(results)
    return results.success

In [ ]:
# Example: Validate a micro-batch using foreachBatch

def process_batch(df, epoch_id):
    pd_df = df.toPandas()
    print(f"Validating batch {epoch_id} with {len(pd_df)} records...")
    valid = validate_with_ge(pd_df)
    if not valid:
        print(f"Batch {epoch_id} failed validation!")
    else:
        print(f"Batch {epoch_id} passed validation.")

# To use in streaming:
clickstream_events.writeStream.foreachBatch(process_batch).start()

In [ ]:
# Example: Persist micro-batch to HDFS

def persist_to_hdfs(df, epoch_id):
    output_path = f"hdfs://dists-hdfs-namenode:8020/user/clickstream/batch_{epoch_id}"
    (
        df.write.mode("append")
        .parquet(output_path)
    )
    print(f"Persisted batch {epoch_id} to {output_path}")

# To use in streaming:
clickstream_events.writeStream.foreachBatch(persist_to_hdfs).start()

In [ ]:
# Combined: Validate and persist micro-batch to HDFS

def validate_and_persist(df, epoch_id):
    pd_df = df.toPandas()
    print(f"Validating batch {epoch_id} with {len(pd_df)} records...")
    valid = validate_with_ge(pd_df)
    if valid:
        output_path = f"hdfs://dists-hdfs-namenode:8020/user/clickstream/batch_{epoch_id}"
        (
            df.write.mode("append")
            .parquet(output_path)
        )
        print(f"Persisted batch {epoch_id} to {output_path}")
    else:
        print(f"Batch {epoch_id} failed validation and was not persisted.")

# To use in streaming:
# clickstream_events.writeStream.foreachBatch(validate_and_persist).start()

In [ ]:
# Query data from HDFS after ingestion

def read_clickstream_from_hdfs(batch_id=None):
    base_path = "hdfs://namenode:9000/user/clickstream/"
    if batch_id is not None:
        path = f"{base_path}batch_{batch_id}"
    else:
        path = base_path  # Reads all batches
    df = spark.read.parquet(path)
    df.show(20, truncate=False)
    return df

# Example usage:
# df = read_clickstream_from_hdfs()
# df = read_clickstream_from_hdfs(batch_id=0)